### CRIA A ENGINE 

In [1]:
%pip install sqlalchemy

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import sqlalchemy
from sqlalchemy import create_engine, text

with engine.connect() as conn:
    result = conn.execute(text("select 'hello world'"))
    print(result.all())

[('hello world',)]


In [ ]:
# EXEMPLOS MARCELO 1
"""
from sqlalchemy import Table, select, MetaData

metadata = MetaData()

activity = Table("activity", metadata, autoload_with=engine)

with engine.connect() as conn:
    res = conn.execute(activity.select())
    print(res.all())
"""

[(1, 'nome', 'teste', datetime.datetime(2026, 4, 16, 14, 47, 6, 854197), datetime.datetime(2026, 4, 16, 14, 47, 6, 854197))]


In [ ]:
# EXEMPLOS MARCELO 2

"""
from sqlalchemy.orm import DeclarativeBase, mapped_column, Mapped, Session

class Base(DeclarativeBase): pass

class Atividade(Base):
    __tablename__ = "activity"
    id: Mapped[int] =  mapped_column(primary_key=True)
    name: Mapped[str]
    description: Mapped[str]

with Session(bind=engine) as session:
    atv1 = Atividade(name="nome", description="teste")
    session.add(atv1)
    session.commit()

"""

'\nfrom sqlalchemy.orm import DeclarativeBase, mapped_column, Mapped, Session\n\nclass Base(DeclarativeBase): pass\n\nclass Atividade(Base):\n    __tablename__ = "activity"\n    id: Mapped[int] =  mapped_column(primary_key=True)\n    name: Mapped[str]\n    description: Mapped[str]\n\nwith Session(bind=engine) as session:\n    atv1 = Atividade(name="nome", description="teste")\n    session.add(atv1)\n    session.commit()\n\n'

In [5]:
with engine.connect() as conn:
    result = conn.execute(text("select 'hello world'"))
    print(result.all())

[('hello world',)]


In [9]:
# Fazendo commit das alterações

with engine.connect() as conn:
    #conn.execute(text("CREATE TABLE soma_table (x int, y int)"))
    conn.execute(
        text("INSERT INTO soma_table (x, y) VALUES (:x, :y)"),
        [{"x": 1, "y": 2}, {"x": 2, "y": 4}]
    )
    conn.commit()

### BEGIN

É uma opção mais fluída para criar um commit após o código ser executado.

In [12]:
with engine.begin() as conn:
    conn.execute(
        text("INSERT INTO soma_table (x, y) VALUES (:x, :y)"),
        [{"x": 6, "y": 8},{"x": 9, "y": 10}],
    )

### BUSCANDO LINHAS

In [13]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT x, y FROM soma_table"))
    for row in result:
        print(f"x: {row.x} y: {row.y}")

x: 1 y: 2
x: 2 y: 4
x: 1 y: 2
x: 2 y: 4
x: 6 y: 8
x: 9 y: 10


### ACESSANDO LINHAS COM TUPLAS

In [17]:
with engine.connect() as conn:
 result = conn.execute(text("SELECT x, y FROM soma_table"))
for x, y in result:
    print(f"x: {row.x} y: {row.y}")

x: 9 y: 10
x: 9 y: 10
x: 9 y: 10
x: 9 y: 10
x: 9 y: 10
x: 9 y: 10


### ACESSANDO O INDICE INTEIRO

In [24]:
with engine.connect() as conn:
 result = conn.execute(text("SELECT x, y FROM soma_table"))
for row in result:
    x = row[0]
    y = row[0]
    print(f"X:{row.x} | Y:{row.y}")

X:1 | Y:2
X:2 | Y:4
X:1 | Y:2
X:2 | Y:4
X:6 | Y:8
X:9 | Y:10


### ACESSANDO TODOS OS ATRIBUTOS COM O MÉTODO ALL()

In [25]:
with engine.connect() as conn:
 result = conn.execute(text("SELECT x, y FROM soma_table"))
print(result.all())

[(1, 2), (2, 4), (1, 2), (2, 4), (6, 8), (9, 10)]


In [27]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT x, y FROM soma_table"))

    for row in result:
        y = row.y
        print(f"Row: {row.x} {y}")

Row: 1 2
Row: 2 4
Row: 1 2
Row: 2 4
Row: 6 8
Row: 9 10


### ACESSO POR MAPEAMENTO

In [ ]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT x, y FROM soma_table"))

    for dict_row in result.mappings():
        x = dict_row["x"]
        y = dict_row["y"]
        print(dict_row)

{'x': 1, 'y': 2}
{'x': 2, 'y': 4}
{'x': 1, 'y': 2}
{'x': 2, 'y': 4}
{'x': 6, 'y': 8}
{'x': 9, 'y': 10}


### ENVIO DE PARAMÊTROS

In [32]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT x, y FROM soma_table WHERE y > :y"), {"y": 2})
    for row in result:
        print(f"x: {row.x} y: {row.y}")

x: 2 y: 4
x: 2 y: 4
x: 6 y: 8
x: 9 y: 10


### ENVIO DE MULTIPLOS PARAMETROS

In [ ]:
with engine.connect() as conn:
    conn.execute(
        text("INSERT INTO soma_table (x, y) VALUES (:x, :y)"),
        [{"x": 11, "y": 12}, {"x": 13, "y": 14}],
    )

    conn.commit()

## EXECUTANDO COM SESSOES ORM

In [37]:
from sqlalchemy.orm import Session

stmt = text("SELECT x, y FROM soma_table WHERE y > :y ORDER BY x, y")
with Session(engine) as session:
    result = session.execute(stmt, {"y": 6})
    for row in result:
        print(f"x: {row.x} y:{row.y}")

x: 6 y:8
x: 9 y:10
x: 11 y:12
x: 13 y:14
